# 01 - A tour of the simulator (start here)

**Goal:** understand what the toolkit models, in plain words, in ~15 minutes.

A design has three parts, each set in a YAML file in `configs/`:

| Part | What it is | Config section |
|---|---|---|
| **Mover** | the floating puck: an array of permanent magnets | `mover` |
| **Coils** | flat air-core windings under the surface; current in them pushes on the magnets | `coil` |
| **Tiles** | identical boards of coils, placed edge to edge | `tile`, `grid` |

The one physics idea you need: **force on the mover = K x (coil currents)**. For the mover sitting at
one spot, `K` is a table (6 rows: Fx, Fy, Fz and the three tilting/turning torques; one column
per coil) saying how hard each coil pushes per amp. Double a current -> double its push.
Everything else (how much current to hover, heat, control, tile handoff) is built on `K`.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from maglev_sim import *
from maglev_sim import plotting

In [ ]:
cfg = load_config("../configs/baseline.yaml")
system = build_system(cfg)
print(system.describe())

Top view. Orange = coils that are 'active' for the mover where it is now (darker = more involved). Red magnets point north-up, blue south-up.

In [ ]:
plotting.plot_layout(system);

## The K table at one position
Columns are coils. Only coils near the mover have non-zero entries - coil influence is local
(lit review takeaway), which is what makes separate tiles possible.

In [ ]:
pose = system.pose()            # table centre at the operating gap
K = system.wrench_matrix(pose)
near = np.flatnonzero(np.abs(K).sum(0) > 0)
import pandas as pd
pd.DataFrame(K[:, near], index=AXES, columns=[f"coil {c}" for c in near]).round(4)

## How much current to just float?
`allocate` picks the lowest-power set of currents that makes exactly the force we want
(here: lift = weight, everything else zero).

In [ ]:
from maglev_sim.allocation import allocate
from maglev_sim.handoff import system_weights
a = allocate(K, system.hover_wrench(), system.control_axes, system_weights(system, pose[:2]), system.max_current)
print(f"weight {system.weight:.3f} N -> produced Fz {a.achieved[2]:.3f} N")
print(f"largest coil current {a.max_abs_current:.2f} A, total power {np.sum(a.currents**2)*system.coil.resistance:.2f} W")

## The standard design report
One call gives the numbers the team compares designs by. It samples many mover positions,
because a design that works at the centre can fail between coils.

In [ ]:
rep = evaluate_design(system, per_tile=5)
print(rep.text())

In [ ]:
plotting.plot_report(rep);

**Reading it:** *lift margin* > 1.5 everywhere is comfortable; *hover power* is heat you must get rid
of continuously; *condition number* says how evenly the coils can push in every direction (big =
some direction is expensive). Next: notebook 02 to change a coil yourself.